# 01 – Data Exploration

This notebook explores the MIT-BIH Arrhythmia Database structure, class distribution, and raw signal characteristics.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt
import config
from src.data_loader import ECGDataLoader
from src.utils import setup_logging, plot_ecg_signal

setup_logging('INFO')
print('AAMI classes:', config.AAMI_CLASSES)
print('Total records:', len(config.MITBIH_RECORDS))

In [ ]:
# Download and load a single record
record = '100'
data_dir = os.path.join('..', config.DATA_DIR)
ECGDataLoader.download_record(record, data_dir)
signal, fields, annotation = ECGDataLoader.load_record(record, data_dir)
print(f'Signal shape: {signal.shape}, fs={fields["fs"]} Hz')

In [ ]:
# Plot first 10 seconds of lead 0
fs = fields['fs']
plot_ecg_signal(signal[:10*fs, 0], fs=fs, title=f'Record {record} – Lead MLII')
plt.show()

In [ ]:
# Explore beat annotation distribution
import pandas as pd
from collections import Counter

if annotation is not None:
    symbol_counts = Counter(annotation.symbol)
    # Map to AAMI
    aami_counts = Counter()
    for sym, cnt in symbol_counts.items():
        aami = config.MITBIH_TO_AAMI.get(sym)
        if aami:
            aami_counts[aami] += cnt
    df = pd.DataFrame(list(aami_counts.items()), columns=['Class', 'Count']).sort_values('Count', ascending=False)
    print(df.to_string(index=False))
    df.set_index('Class')['Count'].plot(kind='bar', title='Beat class distribution – Record 100')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
# Patient-aware split preview
from src.evaluator import PatientSplitter
splitter = PatientSplitter(random_state=42)
train_recs, val_recs, test_recs = splitter.split(config.MITBIH_RECORDS)
print(f'Train: {len(train_recs)} | Val: {len(val_recs)} | Test: {len(test_recs)}')
print('Test records:', test_recs)